# Demo: GOES RGB + ERA5-Land Cloud Mask

Set a latitude/longitude box and UTC time range, then run this notebook to download GOES data, orthorectify it, build RGB composites, download ERA5-Land 2 m temperature, apply the temperature-bin RGB cloud mask, and plot the RGB next to the mask.

The default demo is the Colorado domain on **2020-06-30**. You need your own `OPENTOPOGRAPHY_API_KEY` environment variable and Copernicus/CDS credentials in `~/.cdsapirc`.

## 1. User Parameters

Change this cell for your own research domain. Longitudes should be negative for western hemisphere locations.

In [ ]:
from pathlib import Path

# Demo domain: Colorado, June 30, 2020
DOMAIN = "colorado"
GOES = "goes16"
START_DATE = "2020-06-30"
END_DATE = "2020-06-30"

# Bounding box: lon_min, lat_min, lon_max, lat_max
LON_MIN = -109.0
LAT_MIN = 37.0
LON_MAX = -104.0
LAT_MAX = 41.0

# UTC hours to download and mask. Use a short window for the demo.
GOES_HOURS = "18-22"
START_HOUR_UTC = 18
END_HOUR_UTC = 22

# Where outputs should be written. Keep this on scratch/project storage if possible.
BASE_DIR = Path("./demo_output/colorado")

# Set to False if you only want to inspect commands/paths without downloading data.
RUN_WORKFLOW = True
OVERWRITE_MASK = True

# Which timestep to plot at the end. None plots the first available mask timestep.
PLOT_TIME_UTC = "2020-06-30 20:00"

## 2. Imports And Paths

In [ ]:
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

REPO_DIR = Path.cwd()
if not (REPO_DIR / "scripts").exists() and (REPO_DIR.parent / "scripts").exists():
    REPO_DIR = REPO_DIR.parent

SCRIPTS_DIR = REPO_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
THRESHOLD_CSV = REPO_DIR / "thresholds" / "gothic_temp_bin_rgb_thresholds_10c.csv"

DOWNLOAD_SCRIPT = SCRIPTS_DIR / "download-goes.py"
ORTHO_SCRIPT = SCRIPTS_DIR / "batch_ortho.py"
ZARR_SCRIPT = SCRIPTS_DIR / "zarr_v2_days.py"
MASK_SCRIPT = SCRIPTS_DIR / "apply_tempbin_thresholds.py"

RGB_DIR = BASE_DIR / GOES / "rgb_composite"
ERA5_DIR = BASE_DIR / "era5_land" / "t2m_hourly"
MASK_DIR = BASE_DIR / GOES / "cloud_mask_tempbin_10c"
GIF_DIR = BASE_DIR / GOES / "gif_loops_tempbin_10c"

for path in [BASE_DIR, ERA5_DIR, MASK_DIR, GIF_DIR]:
    path.mkdir(parents=True, exist_ok=True)

date_index = pd.date_range(START_DATE, END_DATE, freq="D")
if date_index.empty:
    raise ValueError("START_DATE/END_DATE produced no dates")

print(f"Repository: {REPO_DIR}")
print(f"Dates: {date_index[0].date()} through {date_index[-1].date()}")
print(f"Output base: {BASE_DIR.resolve()}")

## 3. Credential Checks

GOES is public. OpenTopography and ERA5-Land require user-owned credentials.

In [ ]:
if RUN_WORKFLOW:
    if not os.environ.get("OPENTOPOGRAPHY_API_KEY"):
        raise RuntimeError(
            "Set your own OPENTOPOGRAPHY_API_KEY before running this notebook. "
            "Example: export OPENTOPOGRAPHY_API_KEY='<your-key>'"
        )
    cds_config = Path.home() / ".cdsapirc"
    if not cds_config.exists():
        raise RuntimeError(
            "Missing ~/.cdsapirc. Copy example_cdsapirc to ~/.cdsapirc and add your own Copernicus/CDS key."
        )
else:
    print("RUN_WORKFLOW=False, so credential checks are informational only.")

## 4. Helpers

In [ ]:
def run_command(cmd, env=None, cwd=REPO_DIR):
    """Print and run a shell command represented as a list."""
    cmd = [str(part) for part in cmd]
    print("\n$ " + " ".join(cmd))
    if not RUN_WORKFLOW:
        return None
    return subprocess.run(cmd, cwd=cwd, env=env, check=True)


def workflow_env():
    env = os.environ.copy()
    env.update(
        {
            "GOES_HOURS": GOES_HOURS,
            "LON_MIN": str(LON_MIN),
            "LAT_MIN": str(LAT_MIN),
            "LON_MAX": str(LON_MAX),
            "LAT_MAX": str(LAT_MAX),
            "PYTHONPATH": f"{SCRIPTS_DIR}:{env.get('PYTHONPATH', '')}",
        }
    )
    return env


def ymd_parts(timestamp):
    return timestamp.year, timestamp.month, timestamp.day, timestamp.strftime("%Y%m%d")


def dedupe_zarr_timestamps(date_ymd):
    for channel in ("C02", "C05", "C13"):
        zarr_path = BASE_DIR / GOES / channel / f"{GOES}_{channel}_{DOMAIN}_{date_ymd}.zarr"
        if not zarr_path.is_dir():
            print(f"Missing zarr, skipping dedupe: {zarr_path}")
            continue
        ds = xr.open_dataset(zarr_path)
        try:
            if "t" not in ds.coords or ds.indexes["t"].is_unique:
                continue
            keep = ~ds.indexes["t"].duplicated(keep="first")
            tmp = Path(str(zarr_path) + ".tmp")
            if tmp.exists():
                shutil.rmtree(tmp)
            ds.isel(t=keep).to_zarr(tmp, mode="w")
        finally:
            ds.close()
        shutil.rmtree(zarr_path)
        tmp.rename(zarr_path)
        print(f"Deduped {zarr_path}")

## 5. Download GOES Channels

In [ ]:
env = workflow_env()
for ts in date_index:
    year, month, day, date_ymd = ymd_parts(ts)
    for channel in ("C02", "C05", "C13"):
        run_command(
            [
                sys.executable,
                DOWNLOAD_SCRIPT,
                "-B",
                f"noaa-{GOES}",
                "-Y",
                year,
                "-M",
                month,
                "-D",
                day,
                day,
                "-p",
                "ABI-L1b-RadC",
                "-c",
                channel,
                "-b",
                LON_MIN,
                LAT_MIN,
                LON_MAX,
                LAT_MAX,
                "-d",
                BASE_DIR,
            ],
            env=env,
        )

## 6. Orthorectify, Convert To Zarr, And Build RGB

In [ ]:
import utils

for ts in date_index:
    year, month, day, date_ymd = ymd_parts(ts)
    month_dir = BASE_DIR / GOES / str(year) / str(month)

    run_command([sys.executable, ORTHO_SCRIPT, month_dir, DOMAIN], env=env)
    run_command([sys.executable, ZARR_SCRIPT, BASE_DIR, year, month, day, day, GOES, DOMAIN], env=env)

    if RUN_WORKFLOW:
        dedupe_zarr_timestamps(date_ymd)
        utils.goes_rad_to_rgb(str(BASE_DIR / GOES) + "/", date_ymd, GOES, location=DOMAIN)

    rgb_path = RGB_DIR / f"{GOES}_C02_C05_C13_rgb_{DOMAIN}_{date_ymd}.nc"
    print(f"RGB path: {rgb_path}")

## 7. Download ERA5-Land And Apply The Cloud Mask

In [ ]:
mask_paths = []
for ts in date_index:
    _, _, _, date_ymd = ymd_parts(ts)
    rgb_path = RGB_DIR / f"{GOES}_C02_C05_C13_rgb_{DOMAIN}_{date_ymd}.nc"
    cmd = [
        sys.executable,
        MASK_SCRIPT,
        "--rgb-file",
        rgb_path,
        "--threshold-csv",
        THRESHOLD_CSV,
        "--era5-dir",
        ERA5_DIR,
        "--mask-dir",
        MASK_DIR,
        "--gif-dir",
        GIF_DIR,
        "--domain",
        DOMAIN,
        "--start-hour-utc",
        START_HOUR_UTC,
        "--end-hour-utc",
        END_HOUR_UTC,
    ]
    if OVERWRITE_MASK:
        cmd.append("--overwrite")
    run_command(cmd, env=env)
    mask_path = MASK_DIR / f"{rgb_path.stem}_cloud_binary_tempbin10c.nc"
    mask_paths.append(mask_path)
    print(f"Mask path: {mask_path}")

## 8. Plot RGB And Mask

This plots the nearest available timestep to `PLOT_TIME_UTC`, or the first mask timestep if `PLOT_TIME_UTC = None`.

In [ ]:
if not RUN_WORKFLOW:
    raise RuntimeError("Set RUN_WORKFLOW=True and run the workflow cells before plotting outputs.")

plot_date = pd.Timestamp(START_DATE).strftime("%Y%m%d")
rgb_path = RGB_DIR / f"{GOES}_C02_C05_C13_rgb_{DOMAIN}_{plot_date}.nc"
mask_path = MASK_DIR / f"{rgb_path.stem}_cloud_binary_tempbin10c.nc"

with xr.open_dataset(rgb_path) as rgb_ds, xr.open_dataset(mask_path) as mask_ds:
    if PLOT_TIME_UTC is None:
        plot_time = pd.Timestamp(mask_ds["t"].values[0])
    else:
        plot_time = pd.Timestamp(PLOT_TIME_UTC)

    rgb_frame = rgb_ds.sel(t=plot_time, method="nearest")
    mask_frame = mask_ds.sel(t=plot_time, method="nearest")
    actual_time = pd.Timestamp(mask_frame["t"].values)

    rgb_image = np.stack(
        [rgb_frame["red"].values, rgb_frame["green"].values, rgb_frame["blue"].values],
        axis=-1,
    )
    rgb_image = np.clip(np.nan_to_num(rgb_image, nan=0.0), 0.0, 1.0)
    cloud_mask = mask_frame["cloud_binary"].values

    lon = rgb_ds["longitude"].values
    lat = rgb_ds["latitude"].values
    extent = [float(np.nanmin(lon)), float(np.nanmax(lon)), float(np.nanmin(lat)), float(np.nanmax(lat))]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
axes[0].imshow(rgb_image, origin="lower", extent=extent, aspect="auto")
axes[0].set_title(f"GOES RGB\n{actual_time:%Y-%m-%d %H:%M UTC}")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

im = axes[1].imshow(cloud_mask, origin="lower", extent=extent, aspect="auto", vmin=0, vmax=1, cmap="Blues_r")
axes[1].set_title("Temperature-Bin RGB Cloud Mask")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
cbar = fig.colorbar(im, ax=axes[1], ticks=[0, 1], shrink=0.8)
cbar.ax.set_yticklabels(["clear", "cloud"])
plt.show()